In [1]:
import os
import json
import random
import time
import warnings

import numpy as np
import pandas as pd

import pennylane as qml
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support
)

warnings.filterwarnings("ignore")

In [2]:
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

torch.set_default_dtype(torch.float32)

print("PennyLane:", qml.__version__)
print("PyTorch:", torch.__version__)

PennyLane: 0.45.1
PyTorch: 2.13.0+cpu


In [3]:
clinical_file = (
    "clinical_feature_engineered_data.xlsx"
)

quantum_data_file = (
    "quantum_encoded_healthcare_data.xlsx"
)

quantum_model_file = (
    "optimized_variational_quantum_model.pt"
)

integration_results_file = (
    "end_to_end_system_integration_results.xlsx"
)

integration_config_file = (
    "end_to_end_system_configuration.json"
)

deployment_bundle_file = (
    "healthcare_qml_deployment_bundle.pt"
)

required_files = [
    clinical_file,
    quantum_data_file,
    quantum_model_file
]

for file_name in required_files:
    if not os.path.exists(file_name):
        raise FileNotFoundError(
            f"{file_name} was not found."
        )

print("All required integration files are available.")

All required integration files are available.


In [4]:
clinical_data = pd.read_excel(
    clinical_file,
    sheet_name="Feature_Engineered_Data"
)

print("Clinical data:", clinical_data.shape)
print(clinical_data.columns.tolist())

display(clinical_data.head())

Clinical data: (500, 31)
['patient_id', 'age_years', 'sex', 'condition', 'glucose_mg_dl', 'systolic_bp_mmhg', 'diastolic_bp_mmhg', 'cholesterol_mg_dl', 'heart_rate_bpm', 'recorded_allergy', 'family_history', 'adherence_level', 'historical_medication_class', 'recorded_outcome', 'data_quality_status', 'age_group', 'pulse_pressure_mmhg', 'systolic_diastolic_ratio', 'glucose_cholesterol_interaction', 'glucose_age_interaction', 'cholesterol_age_interaction', 'bp_age_interaction', 'allergy_recorded_flag', 'family_history_flag', 'adherence_score', 'glucose_mg_dl_dataset_zscore', 'systolic_bp_mmhg_dataset_zscore', 'diastolic_bp_mmhg_dataset_zscore', 'cholesterol_mg_dl_dataset_zscore', 'heart_rate_bpm_dataset_zscore', 'measurement_deviation_score']


,patient_id,age_years,sex,condition,glucose_mg_dl,systolic_bp_mmhg,diastolic_bp_mmhg,cholesterol_mg_dl,heart_rate_bpm,recorded_allergy,...,bp_age_interaction,allergy_recorded_flag,family_history_flag,adherence_score,glucose_mg_dl_dataset_zscore,systolic_bp_mmhg_dataset_zscore,diastolic_bp_mmhg_dataset_zscore,cholesterol_mg_dl_dataset_zscore,heart_rate_bpm_dataset_zscore,measurement_deviation_score
0,SYN-0001,28,Female,Seasonal Allergy,97,132,85,214,73,None recorded,...,3696,0,0,2,-0.324226,0.254356,0.339338,0.424061,-0.315852,0.331566
1,SYN-0002,80,Female,Acid Reflux,80,123,79,189,59,None recorded,...,9840,0,1,1,-0.809337,-0.237734,-0.210049,-0.216205,-1.546215,0.603908
2,SYN-0003,36,Female,Asthma,86,121,66,192,82,None recorded,...,4356,0,0,1,-0.638121,-0.347087,-1.400387,-0.139373,0.475096,0.600013
3,SYN-0004,21,Male,High Cholesterol,103,120,69,278,85,None recorded,...,2520,0,1,2,-0.153010,-0.401764,-1.125693,2.063142,0.738745,0.896471
4,SYN-0005,58,Male,High Cholesterol,79,111,76,244,64,None recorded,...,6438,0,1,2,-0.837873,-0.893854,-0.484742,1.192380,-1.106799,0.903130


In [5]:
X_train_df = pd.read_excel(
    quantum_data_file,
    sheet_name="X_Train_Quantum"
)

X_test_df = pd.read_excel(
    quantum_data_file,
    sheet_name="X_Test_Quantum"
)

y_train_df = pd.read_excel(
    quantum_data_file,
    sheet_name="Y_Train"
)

y_test_df = pd.read_excel(
    quantum_data_file,
    sheet_name="Y_Test"
)

class_mapping = pd.read_excel(
    quantum_data_file,
    sheet_name="Class_Mapping"
)

X_train = X_train_df.to_numpy(
    dtype=np.float32
)

X_test = X_test_df.to_numpy(
    dtype=np.float32
)

y_train = y_train_df[
    "encoded_target"
].to_numpy(dtype=np.int64)

y_test = y_test_df[
    "encoded_target"
].to_numpy(dtype=np.int64)

quantum_feature_names = (
    X_test_df.columns.tolist()
)

print("Quantum training data:", X_train.shape)
print("Quantum test data:", X_test.shape)

Quantum training data: (400, 4)
Quantum test data: (100, 4)


In [6]:
try:
    model_checkpoint = torch.load(
        quantum_model_file,
        map_location="cpu",
        weights_only=False
    )
except TypeError:
    model_checkpoint = torch.load(
        quantum_model_file,
        map_location="cpu"
    )

selected_circuit = model_checkpoint[
    "circuit_name"
]

selected_template = model_checkpoint[
    "template"
]

selected_layers = int(
    model_checkpoint["layers"]
)

n_qubits = int(
    model_checkpoint["n_qubits"]
)

n_classes = int(
    model_checkpoint["n_classes"]
)

weight_shapes = model_checkpoint[
    "weight_shapes"
]

print("Circuit:", selected_circuit)
print("Template:", selected_template)
print("Layers:", selected_layers)
print("Qubits:", n_qubits)
print("Classes:", n_classes)

Circuit: Basic_2_Layers
Template: basic
Layers: 2
Qubits: 4
Classes: 12


In [7]:
assert X_train.shape[1] == n_qubits
assert X_test.shape[1] == n_qubits

assert X_test.min() >= -np.pi - 1e-5
assert X_test.max() <= np.pi + 1e-5

assert len(np.unique(y_train)) == n_classes

assert np.array_equal(
    np.sort(np.unique(y_train)),
    np.arange(n_classes)
)

print("Model and quantum data are compatible.")

Model and quantum data are compatible.


In [8]:
quantum_device = qml.device(
    "default.qubit",
    wires=n_qubits
)

wires = list(range(n_qubits))

@qml.qnode(
    quantum_device,
    interface="torch",
    diff_method="backprop"
)
def integrated_quantum_circuit(
    inputs,
    weights
):
    qml.AngleEmbedding(
        features=inputs,
        wires=wires,
        rotation="Y"
    )

    if selected_template == "basic":
        qml.BasicEntanglerLayers(
            weights=weights,
            wires=wires,
            rotation=qml.RY
        )

    elif selected_template == "strong":
        qml.StronglyEntanglingLayers(
            weights=weights,
            wires=wires
        )

    else:
        raise ValueError(
            f"Unknown circuit template: "
            f"{selected_template}"
        )

    return [
        qml.expval(qml.PauliZ(wire))
        for wire in wires
    ]

In [9]:
class IntegratedQuantumClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.quantum_layer = qml.qnn.TorchLayer(
            integrated_quantum_circuit,
            weight_shapes
        )

        self.output_layer = nn.Linear(
            n_qubits,
            n_classes
        )

    def forward(self, inputs):
        quantum_features = self.quantum_layer(
            inputs
        )

        return self.output_layer(
            quantum_features
        )


quantum_model = IntegratedQuantumClassifier()

quantum_model.load_state_dict(
    model_checkpoint["model_state_dict"]
)

quantum_model.eval()

print("Optimized quantum model loaded.")

Optimized quantum model loaded.


In [10]:
class_mapping.columns = (
    class_mapping.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

encoded_column = (
    "encoded_label"
    if "encoded_label" in class_mapping.columns
    else "encoded_target"
)

medicine_column = (
    "historical_medication_class"
)

if encoded_column not in class_mapping.columns:
    raise KeyError(
        "Encoded class column was not found."
    )

if medicine_column not in class_mapping.columns:
    raise KeyError(
        "Medication-class column was not found."
    )

class_dictionary = {
    int(encoded_value): str(medicine_class)
    for encoded_value, medicine_class in zip(
        class_mapping[encoded_column],
        class_mapping[medicine_column]
    )
}

assert len(class_dictionary) == n_classes

print(class_dictionary)

{0: 'Acid-suppression class A', 1: 'Acid-suppression class B', 2: 'Antidiabetic class A', 3: 'Antidiabetic class B', 4: 'Antihistamine class A', 5: 'Antihistamine class B', 6: 'Antihypertensive class A', 7: 'Antihypertensive class B', 8: 'Bronchodilator class', 9: 'Controller inhaler class', 10: 'Lipid-lowering class A', 11: 'Lipid-lowering class B'}


In [11]:
required_clinical_columns = [
    "patient_id",
    "age_years",
    "condition",
    "glucose_mg_dl",
    "systolic_bp_mmhg",
    "diastolic_bp_mmhg",
    "cholesterol_mg_dl",
    "heart_rate_bpm",
    "recorded_allergy",
    "family_history",
    "adherence_level",
    "historical_medication_class"
]

missing_columns = [
    column
    for column in required_clinical_columns
    if column not in clinical_data.columns
]

if missing_columns:
    raise KeyError(
        f"Missing clinical columns: {missing_columns}"
    )

disease_medicine_map = (
    clinical_data[
        [
            "condition",
            "historical_medication_class"
        ]
    ]
    .drop_duplicates()
    .groupby("condition")[
        "historical_medication_class"
    ]
    .apply(set)
    .to_dict()
)

for disease, medicine_classes in (
    disease_medicine_map.items()
):
    print(disease, "->", medicine_classes)

Acid Reflux -> {'Acid-suppression class B', 'Acid-suppression class A'}
Asthma -> {'Controller inhaler class', 'Bronchodilator class'}
High Cholesterol -> {'Lipid-lowering class A', 'Lipid-lowering class B'}
Hypertension -> {'Antihypertensive class B', 'Antihypertensive class A'}
Seasonal Allergy -> {'Antihistamine class B', 'Antihistamine class A'}
Type 2 Diabetes -> {'Antidiabetic class A', 'Antidiabetic class B'}


In [12]:
clinical_indices = np.arange(
    len(clinical_data)
)

_, reconstructed_test_indices = (
    train_test_split(
        clinical_indices,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=clinical_data[
            "historical_medication_class"
        ]
    )
)

test_patient_profiles = (
    clinical_data
    .iloc[reconstructed_test_indices]
    .reset_index(drop=True)
)

print(
    "Recovered patient profiles:",
    test_patient_profiles.shape
)

assert len(test_patient_profiles) == len(X_test)

Recovered patient profiles: (100, 31)


In [13]:
expected_test_classes = (
    test_patient_profiles[
        "historical_medication_class"
    ]
    .astype(str)
    .str.strip()
    .to_numpy()
)

quantum_test_classes = np.array([
    class_dictionary[
        int(encoded_target)
    ]
    for encoded_target in y_test
])

alignment_matches = (
    expected_test_classes
    == quantum_test_classes
)

print(
    "Aligned records:",
    alignment_matches.sum(),
    "out of",
    len(alignment_matches)
)

if not alignment_matches.all():
    raise ValueError(
        "Patient alignment failed. Do not continue. "
        "Patient IDs must be exported directly with "
        "X_Test_Quantum during feature preparation."
    )

print("Patient alignment verified successfully.")

Aligned records: 100 out of 100
Patient alignment verified successfully.


In [14]:
def assess_patient_risk(
    patient_profile
):
    risk_reasons = []
    risk_score = 0

    glucose = float(
        patient_profile["glucose_mg_dl"]
    )

    systolic = float(
        patient_profile["systolic_bp_mmhg"]
    )

    diastolic = float(
        patient_profile["diastolic_bp_mmhg"]
    )

    cholesterol = float(
        patient_profile["cholesterol_mg_dl"]
    )

    heart_rate = float(
        patient_profile["heart_rate_bpm"]
    )

    age = float(
        patient_profile["age_years"]
    )

    allergy = str(
        patient_profile["recorded_allergy"]
    ).strip()

    adherence = str(
        patient_profile["adherence_level"]
    ).lower().strip()

    if glucose < 54 or glucose >= 300:
        risk_score += 4
        risk_reasons.append(
            "Critical glucose review"
        )

    elif glucose >= 200:
        risk_score += 2
        risk_reasons.append(
            "High glucose review"
        )

    if systolic >= 180 or diastolic >= 120:
        risk_score += 4
        risk_reasons.append(
            "Critical blood-pressure review"
        )

    elif systolic < 90:
        risk_score += 3
        risk_reasons.append(
            "Low blood-pressure review"
        )

    if cholesterol >= 240:
        risk_score += 2
        risk_reasons.append(
            "High cholesterol review"
        )

    if heart_rate < 50 or heart_rate > 120:
        risk_score += 3
        risk_reasons.append(
            "Heart-rate review"
        )

    if age >= 65:
        risk_score += 1
        risk_reasons.append(
            "Older-adult medication review"
        )

    no_allergy_values = {
        "",
        "none",
        "none recorded",
        "no",
        "no known allergy",
        "no known allergies",
        "unknown",
        "nan"
    }

    allergy_review = (
        allergy.lower()
        not in no_allergy_values
    )

    if allergy_review:
        risk_score += 2
        risk_reasons.append(
            "Recorded allergy requires verification"
        )

    if adherence in [
        "low",
        "poor",
        "non-adherent",
        "nonadherent"
    ]:
        risk_score += 1
        risk_reasons.append(
            "Low adherence"
        )

    if risk_score >= 4:
        risk_level = "High review"

    elif risk_score >= 2:
        risk_level = "Moderate review"

    else:
        risk_level = "Routine review"

    return {
        "risk_score": risk_score,
        "risk_level": risk_level,
        "allergy_review": allergy_review,
        "risk_reasons": (
            "; ".join(risk_reasons)
            if risk_reasons
            else "No rule-based risk flags"
        )
    }

In [15]:
class HealthcareQMLIntegrationSystem:
    def __init__(
        self,
        model,
        class_dictionary,
        disease_medicine_map
    ):
        self.model = model
        self.class_dictionary = (
            class_dictionary
        )
        self.disease_medicine_map = (
            disease_medicine_map
        )

    def predict_probabilities(
        self,
        encoded_features
    ):
        features = np.asarray(
            encoded_features,
            dtype=np.float32
        )

        if features.ndim == 1:
            features = features.reshape(1, -1)

        if features.shape[1] != n_qubits:
            raise ValueError(
                f"Expected {n_qubits} encoded "
                f"features, received "
                f"{features.shape[1]}."
            )

        if (
            features.min() < -np.pi - 1e-5
            or features.max() > np.pi + 1e-5
        ):
            raise ValueError(
                "Encoded quantum angles must be "
                "between -pi and pi."
            )

        input_tensor = torch.tensor(
            features,
            dtype=torch.float32
        )

        self.model.eval()

        with torch.no_grad():
            logits = self.model(input_tensor)

            probabilities = torch.softmax(
                logits,
                dim=1
            ).cpu().numpy()

        return probabilities

    def recommend(
        self,
        encoded_features,
        patient_profile,
        top_k=3
    ):
        probabilities = (
            self.predict_probabilities(
                encoded_features
            )[0]
        )

        ranked_class_ids = np.argsort(
            probabilities
        )[::-1]

        disease = str(
            patient_profile["condition"]
        )

        compatible_classes = (
            self.disease_medicine_map.get(
                disease,
                set()
            )
        )

        risk_result = assess_patient_risk(
            patient_profile
        )

        entropy = -np.sum(
            np.clip(
                probabilities,
                1e-12,
                1.0
            )
            * np.log(
                np.clip(
                    probabilities,
                    1e-12,
                    1.0
                )
            )
        )

        normalized_entropy = (
            entropy / np.log(n_classes)
        )

        sorted_probabilities = np.sort(
            probabilities
        )

        probability_margin = (
            sorted_probabilities[-1]
            - sorted_probabilities[-2]
        )

        all_ranked_results = []
        eligible_results = []

        for original_rank, class_id in enumerate(
            ranked_class_ids,
            start=1
        ):
            medication_class = (
                self.class_dictionary[
                    int(class_id)
                ]
            )

            compatible = (
                medication_class
                in compatible_classes
            )

            result = {
                "original_rank":
                    original_rank,
                "encoded_class":
                    int(class_id),
                "medication_class":
                    medication_class,
                "model_score":
                    float(
                        probabilities[class_id]
                    ),
                "disease_compatible":
                    compatible
            }

            all_ranked_results.append(result)

            if compatible:
                eligible_results.append(
                    result.copy()
                )

        final_results = (
            eligible_results[:top_k]
        )

        for final_rank, result in enumerate(
            final_results,
            start=1
        ):
            result["safety_adjusted_rank"] = (
                final_rank
            )

        requires_review = (
            risk_result["risk_level"]
            != "Routine review"
            or risk_result[
                "allergy_review"
            ]
            or normalized_entropy >= 0.80
        )

        return {
            "probabilities": probabilities,
            "raw_predicted_class":
                int(ranked_class_ids[0]),
            "raw_predicted_medication":
                self.class_dictionary[
                    int(ranked_class_ids[0])
                ],
            "raw_confidence":
                float(
                    probabilities[
                        ranked_class_ids[0]
                    ]
                ),
            "normalized_entropy":
                float(normalized_entropy),
            "probability_margin":
                float(probability_margin),
            "risk_result":
                risk_result,
            "requires_professional_review":
                requires_review,
            "recommendations":
                final_results,
            "all_ranked_results":
                all_ranked_results
        }

In [16]:
integrated_system = (
    HealthcareQMLIntegrationSystem(
        model=quantum_model,
        class_dictionary=class_dictionary,
        disease_medicine_map=
            disease_medicine_map
    )
)

print(
    "Healthcare QML integration system initialized."
)

Healthcare QML integration system initialized.


In [17]:
example_index = 0

example_patient = (
    test_patient_profiles
    .iloc[example_index]
)

example_quantum_features = (
    X_test[example_index]
)

example_result = (
    integrated_system.recommend(
        encoded_features=
            example_quantum_features,
        patient_profile=
            example_patient,
        top_k=3
    )
)

print(
    "Patient ID:",
    example_patient["patient_id"]
)

print(
    "Condition:",
    example_patient["condition"]
)

print(
    "Raw prediction:",
    example_result[
        "raw_predicted_medication"
    ]
)

print(
    "Raw confidence:",
    round(
        example_result[
            "raw_confidence"
        ],
        4
    )
)

print(
    "Risk level:",
    example_result[
        "risk_result"
    ]["risk_level"]
)

print(
    "Review required:",
    example_result[
        "requires_professional_review"
    ]
)

print("\nSafety-filtered recommendations:")

for recommendation in example_result[
    "recommendations"
]:
    print(recommendation)

Patient ID: SYN-0049
Condition: Asthma
Raw prediction: Antihistamine class A
Raw confidence: 0.2169
Risk level: Routine review
Review required: True

Safety-filtered recommendations:
{'original_rank': 2, 'encoded_class': 9, 'medication_class': 'Controller inhaler class', 'model_score': 0.15269699692726135, 'disease_compatible': True, 'safety_adjusted_rank': 1}
{'original_rank': 6, 'encoded_class': 8, 'medication_class': 'Bronchodilator class', 'model_score': 0.12309107184410095, 'disease_compatible': True, 'safety_adjusted_rank': 2}


In [18]:
patient_output_rows = []
ranking_output_rows = []

integration_start_time = time.time()

for test_index in range(len(X_test)):
    patient_profile = (
        test_patient_profiles.iloc[
            test_index
        ]
    )

    result = integrated_system.recommend(
        encoded_features=X_test[test_index],
        patient_profile=patient_profile,
        top_k=3
    )

    eligible_recommendations = result[
        "recommendations"
    ]

    if eligible_recommendations:
        final_recommendation = (
            eligible_recommendations[0]
        )

        final_medication_class = (
            final_recommendation[
                "medication_class"
            ]
        )

        final_score = (
            final_recommendation[
                "model_score"
            ]
        )

        final_encoded_class = (
            final_recommendation[
                "encoded_class"
            ]
        )

    else:
        final_medication_class = (
            "No compatible recommendation"
        )

        final_score = np.nan
        final_encoded_class = -1

    actual_medication_class = (
        class_dictionary[
            int(y_test[test_index])
        ]
    )

    patient_output_rows.append({
        "patient_id":
            patient_profile["patient_id"],
        "condition":
            patient_profile["condition"],
        "actual_encoded_target":
            int(y_test[test_index]),
        "actual_medication_class":
            actual_medication_class,
        "raw_predicted_encoded_class":
            result["raw_predicted_class"],
        "raw_predicted_medication_class":
            result[
                "raw_predicted_medication"
            ],
        "raw_prediction_confidence":
            result["raw_confidence"],
        "normalized_entropy":
            result["normalized_entropy"],
        "probability_margin":
            result["probability_margin"],
        "risk_score":
            result["risk_result"][
                "risk_score"
            ],
        "risk_level":
            result["risk_result"][
                "risk_level"
            ],
        "risk_reasons":
            result["risk_result"][
                "risk_reasons"
            ],
        "final_recommended_encoded_class":
            final_encoded_class,
        "final_recommended_medication_class":
            final_medication_class,
        "final_recommendation_score":
            final_score,
        "requires_professional_review":
            result[
                "requires_professional_review"
            ],
        "recommendation_available":
            len(eligible_recommendations) > 0,
        "final_recommendation_matches_recorded":
            (
                final_medication_class
                == actual_medication_class
            )
    })

    for recommendation in eligible_recommendations:
        ranking_output_rows.append({
            "patient_id":
                patient_profile["patient_id"],
            "condition":
                patient_profile["condition"],
            "safety_adjusted_rank":
                recommendation[
                    "safety_adjusted_rank"
                ],
            "encoded_class":
                recommendation[
                    "encoded_class"
                ],
            "recommended_medication_class":
                recommendation[
                    "medication_class"
                ],
            "model_score":
                recommendation[
                    "model_score"
                ],
            "risk_level":
                result["risk_result"][
                    "risk_level"
                ],
            "requires_professional_review":
                result[
                    "requires_professional_review"
                ]
        })

integration_seconds = (
    time.time() - integration_start_time
)

integrated_patient_results = pd.DataFrame(
    patient_output_rows
)

integrated_ranking_results = pd.DataFrame(
    ranking_output_rows
)

print(
    "Integrated patient results:",
    integrated_patient_results.shape
)

print(
    "Integrated ranking results:",
    integrated_ranking_results.shape
)

print(
    "Integration time:",
    round(integration_seconds, 4),
    "seconds"
)

Integrated patient results: (100, 18)
Integrated ranking results: (200, 8)
Integration time: 1.5986 seconds


In [19]:
raw_predictions = (
    integrated_patient_results[
        "raw_predicted_encoded_class"
    ].to_numpy(dtype=np.int64)
)

raw_accuracy = accuracy_score(
    y_test,
    raw_predictions
)

raw_balanced_accuracy = (
    balanced_accuracy_score(
        y_test,
        raw_predictions
    )
)

_, _, raw_macro_f1, _ = (
    precision_recall_fscore_support(
        y_test,
        raw_predictions,
        average="macro",
        zero_division=0
    )
)

print("Raw accuracy:",
      round(raw_accuracy, 4))

print("Raw balanced accuracy:",
      round(raw_balanced_accuracy, 4))

print("Raw macro F1:",
      round(raw_macro_f1, 4))

Raw accuracy: 0.27
Raw balanced accuracy: 0.2503
Raw macro F1: 0.198


In [20]:
recommendation_coverage = (
    integrated_patient_results[
        "recommendation_available"
    ].mean()
)

final_recommendation_accuracy = (
    integrated_patient_results[
        "final_recommendation_matches_recorded"
    ].mean()
)

review_rate = (
    integrated_patient_results[
        "requires_professional_review"
    ].mean()
)

system_metrics = pd.DataFrame({
    "metric": [
        "Raw model accuracy",
        "Raw model balanced accuracy",
        "Raw model macro F1",
        "Recommendation coverage",
        "Final recommendation match rate",
        "Professional review rate",
        "Patients processed",
        "Integration seconds",
        "Records per second"
    ],
    "value": [
        raw_accuracy,
        raw_balanced_accuracy,
        raw_macro_f1,
        recommendation_coverage,
        final_recommendation_accuracy,
        review_rate,
        len(integrated_patient_results),
        integration_seconds,
        (
            len(integrated_patient_results)
            / integration_seconds
        )
    ]
})

display(system_metrics)

,metric,value
0,Raw model accuracy,0.270000
1,Raw model balanced accuracy,0.250331
2,Raw model macro F1,0.197958
3,Recommendation coverage,1.000000
4,Final recommendation match rate,0.490000
5,Professional review rate,0.790000
6,Patients processed,100.000000
7,Integration seconds,1.598563
8,Records per second,62.556194


In [21]:
integration_checks = []

def add_check(
    check_name,
    passed,
    details
):
    integration_checks.append({
        "check": check_name,
        "status": (
            "PASS" if passed else "FAIL"
        ),
        "details": details
    })


add_check(
    "Required files",
    all(
        os.path.exists(file_name)
        for file_name in required_files
    ),
    "All required files available"
)

add_check(
    "Data-model feature compatibility",
    X_test.shape[1] == n_qubits,
    f"{X_test.shape[1]} features, "
    f"{n_qubits} qubits"
)

add_check(
    "Patient alignment",
    alignment_matches.all(),
    f"{alignment_matches.sum()}/"
    f"{len(alignment_matches)} aligned"
)

add_check(
    "Class mapping",
    len(class_dictionary) == n_classes,
    f"{len(class_dictionary)} classes"
)

add_check(
    "Probability validity",
    integrated_patient_results[
        "raw_prediction_confidence"
    ].between(0, 1).all(),
    "All confidence values between 0 and 1"
)

add_check(
    "Patient uniqueness",
    not integrated_patient_results[
        "patient_id"
    ].duplicated().any(),
    "No duplicate test patient IDs"
)

add_check(
    "Disease-compatible output",
    all(
        row[
            "final_recommended_medication_class"
        ]
        in disease_medicine_map.get(
            row["condition"],
            set()
        )
        or row[
            "final_recommended_medication_class"
        ] == "No compatible recommendation"
        for _, row in (
            integrated_patient_results.iterrows()
        )
    ),
    "Final outputs follow observed disease mapping"
)

integration_check_report = pd.DataFrame(
    integration_checks
)

display(integration_check_report)

if (
    integration_check_report["status"]
    == "FAIL"
).any():
    raise RuntimeError(
        "One or more integration checks failed."
    )

print("All end-to-end integration checks passed.")

,check,status,details
0,Required files,PASS,All required files available
1,Data-model feature compatibility,PASS,"4 features, 4 qubits"
2,Patient alignment,PASS,100/100 aligned
3,Class mapping,PASS,12 classes
4,Probability validity,PASS,All confidence values between 0 and 1
5,Patient uniqueness,PASS,No duplicate test patient IDs
6,Disease-compatible output,PASS,Final outputs follow observed disease mapping


All end-to-end integration checks passed.


In [22]:
system_manifest = {
    "system_name": (
        "Personalized Healthcare QML "
        "Recommendation System"
    ),
    "system_version": "1.0.0",
    "model_file": quantum_model_file,
    "model_type": (
        "Optimized Hybrid "
        "Quantum-Classical Classifier"
    ),
    "quantum_circuit":
        selected_circuit,
    "quantum_template":
        selected_template,
    "quantum_layers":
        selected_layers,
    "number_of_qubits":
        n_qubits,
    "number_of_classes":
        n_classes,
    "input_feature_names":
        quantum_feature_names,
    "input_angle_range": [
        -float(np.pi),
        float(np.pi)
    ],
    "recommendations_per_patient": 3,
    "safety_layer": (
        "Disease compatibility and "
        "rule-based review flags"
    ),
    "uncertainty_measure": (
        "Normalized predictive entropy"
    ),
    "patient_alignment_verified": True,
    "raw_input_supported": False,
    "raw_input_limitation": (
        "Production inference requires the exact "
        "saved feature engineering, feature selection, "
        "PCA and angle-scaling pipeline"
    ),
    "clinical_status": (
        "Synthetic research prototype; "
        "not clinically validated"
    ),
    "random_state": RANDOM_STATE
}

display(
    pd.DataFrame(
        list(system_manifest.items()),
        columns=["property", "value"]
    )
)

,property,value
0,system_name,Personalized Healthcare QML Recommendation System
1,system_version,1.0.0
2,model_file,optimized_variational_quantum_model.pt
3,model_type,Optimized Hybrid Quantum-Classical Classifier
4,quantum_circuit,Basic_2_Layers
5,quantum_template,basic
6,quantum_layers,2
7,number_of_qubits,4
8,number_of_classes,12
9,input_feature_names,"[qubit_angle_1, qubit_angle_2, qubit_angle_3, ..."


In [23]:
torch.save(
    {
        "system_manifest":
            system_manifest,
        "model_state_dict":
            quantum_model.state_dict(),
        "circuit_name":
            selected_circuit,
        "template":
            selected_template,
        "layers":
            selected_layers,
        "n_qubits":
            n_qubits,
        "n_classes":
            n_classes,
        "weight_shapes":
            weight_shapes,
        "class_mapping":
            class_dictionary,
        "disease_medicine_map": {
            disease: sorted(
                medicine_classes
            )
            for disease, medicine_classes
            in disease_medicine_map.items()
        },
        "quantum_feature_names":
            quantum_feature_names
    },
    deployment_bundle_file
)

print(
    "Deployment bundle saved:",
    deployment_bundle_file
)

Deployment bundle saved: healthcare_qml_deployment_bundle.pt


In [24]:
with open(
    integration_config_file,
    "w"
) as file:
    json.dump(
        system_manifest,
        file,
        indent=4
    )

with pd.ExcelWriter(
    integration_results_file,
    engine="openpyxl"
) as writer:

    integrated_patient_results.to_excel(
        writer,
        sheet_name="Patient_Results",
        index=False
    )

    integrated_ranking_results.to_excel(
        writer,
        sheet_name="Ranked_Recommendations",
        index=False
    )

    system_metrics.to_excel(
        writer,
        sheet_name="System_Metrics",
        index=False
    )

    integration_check_report.to_excel(
        writer,
        sheet_name="Integration_Checks",
        index=False
    )

    class_mapping.to_excel(
        writer,
        sheet_name="Class_Mapping",
        index=False
    )

    pd.DataFrame([
        {
            "condition": disease,
            "medication_class":
                medicine_class
        }
        for disease, medicine_classes
        in disease_medicine_map.items()
        for medicine_class
        in sorted(medicine_classes)
    ]).to_excel(
        writer,
        sheet_name="Disease_Medicine_Map",
        index=False
    )

    pd.DataFrame(
        list(system_manifest.items()),
        columns=["property", "value"]
    ).to_excel(
        writer,
        sheet_name="System_Manifest",
        index=False
    )

print(
    "Integration results saved:",
    integration_results_file
)

print(
    "Configuration saved:",
    integration_config_file
)

Integration results saved: end_to_end_system_integration_results.xlsx
Configuration saved: end_to_end_system_configuration.json


In [25]:
try:
    reloaded_bundle = torch.load(
        deployment_bundle_file,
        map_location="cpu",
        weights_only=False
    )
except TypeError:
    reloaded_bundle = torch.load(
        deployment_bundle_file,
        map_location="cpu"
    )

required_bundle_keys = [
    "system_manifest",
    "model_state_dict",
    "template",
    "layers",
    "n_qubits",
    "n_classes",
    "weight_shapes",
    "class_mapping",
    "disease_medicine_map"
]

missing_bundle_keys = [
    key
    for key in required_bundle_keys
    if key not in reloaded_bundle
]

if missing_bundle_keys:
    raise KeyError(
        f"Deployment bundle is missing: "
        f"{missing_bundle_keys}"
    )

print("Deployment bundle reload successful.")

Deployment bundle reload successful.


In [26]:
output_files = [
    deployment_bundle_file,
    integration_results_file,
    integration_config_file
]

for file_name in output_files:
    print(
        file_name,
        "exists:",
        os.path.exists(file_name)
    )

saved_workbook = pd.ExcelFile(
    integration_results_file
)

print("\nSaved sheets:")
print(saved_workbook.sheet_names)

print("\nEnd-to-end system status: PASS")

print(
    "Patients processed:",
    len(integrated_patient_results)
)

print(
    "Recommendation coverage:",
    round(recommendation_coverage, 4)
)

print(
    "Professional review rate:",
    round(review_rate, 4)
)

print(
    "Raw model accuracy:",
    round(raw_accuracy, 4)
)

print(
    "Raw model macro F1:",
    round(raw_macro_f1, 4)
)

healthcare_qml_deployment_bundle.pt exists: True
end_to_end_system_integration_results.xlsx exists: True
end_to_end_system_configuration.json exists: True

Saved sheets:
['Patient_Results', 'Ranked_Recommendations', 'System_Metrics', 'Integration_Checks', 'Class_Mapping', 'Disease_Medicine_Map', 'System_Manifest']

End-to-end system status: PASS
Patients processed: 100
Recommendation coverage: 1.0
Professional review rate: 0.79
Raw model accuracy: 0.27
Raw model macro F1: 0.198
